# LLM serving benchmark: vLLM vs SGLang on one GPU

**First:** Runtime > Change runtime type > **T4 GPU** (or L4 / A100 if you have Colab Pro). Then Runtime > Run all.

Same model, same context length, same GPU memory budget, same prompts. Each engine gets its own
virtualenv because vLLM and SGLang pin different PyTorch versions.

TensorRT-LLM is run separately in NVIDIA's container on an L4 or newer GPU (see README).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!git clone -q https://github.com/Nirmitdagli/llm-serving-bench.git
%cd llm-serving-bench
!pip install -q uv

## 1. Install each engine in its own environment (about 5 minutes each)

In [ ]:
!uv venv -q /envs/vllm   && uv pip install -q --python /envs/vllm/bin/python   vllm aiohttp
!uv venv -q /envs/sglang && uv pip install -q --python /envs/sglang/bin/python "sglang[all]" aiohttp

## 2. Full precision (fp16)
Qwen2.5-1.5B-Instruct fits a T4 (16 GB) with room for KV cache.
On a T4, SGLang may need the Triton attention backend; that flag is set below.

In [ ]:
!PATH=/envs/vllm/bin:$PATH   ENGINES=vllm   MODEL=Qwen/Qwen2.5-1.5B-Instruct TAG=fp16 REPORT_PY=true bash run_suite.sh
!PATH=/envs/sglang/bin:$PATH ENGINES=sglang MODEL=Qwen/Qwen2.5-1.5B-Instruct TAG=fp16 REPORT_PY=true     SGLANG_EXTRA="--attention-backend triton" bash run_suite.sh

## 3. 4-bit quantized (AWQ), same engines

In [ ]:
!PATH=/envs/vllm/bin:$PATH   ENGINES=vllm   MODEL=Qwen/Qwen2.5-1.5B-Instruct-AWQ TAG=awq-int4 REPORT_PY=true bash run_suite.sh
!PATH=/envs/sglang/bin:$PATH ENGINES=sglang MODEL=Qwen/Qwen2.5-1.5B-Instruct-AWQ TAG=awq-int4 REPORT_PY=true     SGLANG_EXTRA="--attention-backend triton" bash run_suite.sh

## 4. Report: copy these tables into README.md

In [ ]:
!python report.py results
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("results/*.png")):
    display(Image(p))

If an engine fails, its log is in `logs/<engine>-<tag>.log`. Run `!tail -50 logs/sglang-fp16.log`
and paste the error into Claude before changing anything.

Download `results/` (Files panel on the left) and commit it to the repo with the README tables.

In [ ]:
!zip -qr results.zip results logs && ls -la results